# TENSORIX v2: Distillation checks and runs (Bhuvi)

Run cells **one at a time, top to bottom**. Do not use "Run all". Set the runtime to **T4 GPU** first.
Never run two GPU jobs at once (for example a check while a training run is going).
Rerun Cell 1 after every disconnect.

## Cell 1: Setup (run first in every session)

In [ ]:
import os, torch
BRANCH = "feature/kd-v2"    # your pushed branch; change to "main" after the PR is merged
REPO   = "https://github.com/sunidhishekar-create/tensorix-kd-slm.git"
DRIVE  = "/content/drive/MyDrive/tensorix/v2"
TEACHER = "HuggingFaceTB/SmolLM2-360M"

from google.colab import drive
drive.mount("/content/drive")

%cd /content
if not os.path.exists("/content/tensorix-kd-slm"):
    !git clone {REPO}
%cd /content/tensorix-kd-slm
!git fetch origin && git checkout {BRANCH} && git pull origin {BRANCH}
!git log --oneline -5
!pip install -q -r requirements.txt

os.makedirs("/content/data", exist_ok=True)
if not os.path.exists("/content/data/train.bin"):
    !cp {DRIVE}/data/* /content/data/
!ls -lh /content/data
need = ["train.bin", "val_c4.bin", "val_ts.bin", "meta.json"]
missing = [f for f in need if not os.path.exists(f"/content/data/{f}")]
assert not missing, f"missing in /content/data: {missing}"
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU, then rerun Cell 1."
print("GPU:", torch.cuda.get_device_name(0), "| setup OK")

## Cell 2: Unit tests of the KD loss (about 1 minute)
Expected: every line says OK and the last line says all tests passed.

In [ ]:
%cd /content/tensorix-kd-slm
!python src/distill.py --selftest

## Cell 3: Does my teacher loader reproduce the README numbers?
The README says SmolLM2-360M scores **C4 loss 3.047** and **TinyStories loss 1.907** on the fixed validation windows.
Expected here: within about 0.01 of both. If not, STOP and send me the output.

In [ ]:
%cd /content/tensorix-kd-slm
!python src/distill.py --check_teacher --data_dir /content/data --teacher {TEACHER}

## Cell 4: fp16 teacher versus fp32 teacher
Expected: no NaN or inf, a small mean KL (well below 1e-2 at T=1, 2 and 4) and top-1 agreement above about 0.95.

In [ ]:
%cd /content/tensorix-kd-slm
!python src/distill.py --check_fp16 --data_dir /content/data --teacher {TEACHER}

## Cell 5: Speed and memory test (100 steps each, about 5 to 8 minutes in total)
README numbers for comparison: baseline 535 ms/step. The README's KD estimate is about 3.3x that (about 1,800 ms/step); this test gives the real number.
Expected: both runs finish, the KD lines show `ce` and `kl`, and peak memory stays below 14 GiB. If you get CUDA out of memory, rerun the KD line with `--grad_accum 4`.

In [ ]:
import subprocess, sys
def run(cmd):
    """Run a command and stream its output into the notebook (no {braces} expansion needed)."""
    env = dict(os.environ, PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    for line in p.stdout:
        print(line, end="")
    p.wait()
    print(f"[exit code {p.returncode}]")

%cd /content/tensorix-kd-slm
!rm -rf /content/speedtest_base /content/speedtest_kd
common = ["--data_dir", "/content/data", "--max_steps", "100", "--eval_interval", "100", "--save_interval", "100"]
print("=== BASELINE, 100 steps ===")
run([sys.executable, "src/train.py", "--out_dir", "/content/speedtest_base"] + common)
print("=== KD (T=2, alpha=0.5), 100 steps ===")
run([sys.executable, "src/train.py", "--out_dir", "/content/speedtest_kd"] + common +
    ["--teacher", TEACHER, "--kd_alpha", "0.5", "--kd_temp", "2"])

## Cell 6: Real KD run, SHORT tier (5,000 steps)
Edit `KD_T` and `RUN_NAME` (names: `kd_short_T1`, `kd_short_T2`, `kd_short_T4`). Alpha stays 0.5 for the whole sweep.
Do NOT change batch size, block size, steps or eval settings. Rerun Cell 1 and then this cell unchanged after a disconnect: it resumes.
Run `device=cuda` check: the first log line must say `device=cuda`.
This cell defines its own `run()` helper, so after a disconnect you only need to rerun Cell 1 and then this cell.


In [ ]:
import subprocess, sys
def run(cmd):
    """Run a command and stream its output into the notebook."""
    env = dict(os.environ, PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    for line in p.stdout:
        print(line, end="")
    p.wait()
    print(f"[exit code {p.returncode}]")

KD_T     = 2
KD_ALPHA = 0.5
RUN_NAME = "kd_short_T2"        # NEW name for every run; reusing a name RESUMES that run
MAX_STEPS = 5000                # SHORT tier. Never change this inside the sweep.
OUT = f"{DRIVE}/checkpoints/{RUN_NAME}"
if os.path.exists(f"{OUT}/ckpt.pt"):
    print(f"Found a checkpoint in {OUT}: this run will RESUME.")
%cd /content/tensorix-kd-slm
run([sys.executable, "src/train.py", "--data_dir", "/content/data", "--out_dir", OUT,
     "--max_steps", str(MAX_STEPS), "--eval_interval", "250", "--save_interval", "500",
     "--teacher", TEACHER, "--kd_alpha", str(KD_ALPHA), "--kd_temp", str(KD_T)])

## Cell 7: Compare with the baseline (log tables and curves)
Add finished KD runs to `RUNS`. Always compare with `baseline_short` for SHORT runs.
Baseline_short final: C4 ppl 81.2, TinyStories ppl 6.26 (README).

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
RUNS = ["baseline_short", "kd_short_T1", "kd_short_T2", "kd_short_T4"]
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for r in RUNS:
    p = f"{DRIVE}/checkpoints/{r}/log.csv"
    if not os.path.exists(p):
        print("missing:", p); continue
    df = pd.read_csv(p)
    print(r); print(df.tail(3).to_string(index=False)); print()
    ax[0].plot(df.step, df.ppl_c4, marker="o", label=r)
    ax[1].plot(df.step, df.ppl_ts, marker="o", label=r)
for a, name in zip(ax, ["C4", "TinyStories"]):
    a.set(title=f"Validation perplexity: {name}", xlabel="step", ylabel="perplexity", yscale="log")
    a.grid(alpha=0.3); a.legend()
os.makedirs(f"{DRIVE}/results", exist_ok=True)
plt.savefig(f"{DRIVE}/results/val_ppl_kd.png", dpi=150, bbox_inches="tight")
plt.show()

## Cell 8: Score finished KD runs with evaluate.py and compute the retention metric
Only run this when no training run is using the GPU.
Consistency check: `val_c4_loss` and `val_ts_loss` from the JSON must match the last row of the run's `log.csv` to about 3 decimals. If they do not, stop and tell Sunidhi.
Retention (gap closed) is meaningful on **C4 only** (on TinyStories the teacher is worse than the student).

In [ ]:
import json
EVAL_DIR  = f"{DRIVE}/results/eval"
EVAL_KD   = ["kd_short_T2"]            # add kd_short_T1, kd_short_T4 when finished
%cd /content/tensorix-kd-slm
for name in EVAL_KD:
    !python src/evaluate.py --data_dir /content/data --ckpt {DRIVE}/checkpoints/{name}/best.pt --name {name} --out_dir {EVAL_DIR}

def L(name, key):
    return json.load(open(f"{EVAL_DIR}/{name}.json"))[key]

try:
    b, t = L("baseline_short", "val_c4_loss"), L("teacher_base", "val_c4_loss")
    print(f"C4 loss: baseline_short {b:.4f} | teacher {t:.4f}")
    for name in EVAL_KD:
        d = L(name, "val_c4_loss")
        print(f"{name}: C4 loss {d:.4f} | TS loss {L(name, 'val_ts_loss'):.4f} | gap closed on C4 = {(b - d) / (b - t):.3f}")
except FileNotFoundError as e:
    print("missing JSON:", e, "(run Cell 5/6 of baseline_train.ipynb with EVAL_TEACHER = True first)")